# Week 6 - Feature Engineering And Pipeline Selection

**Business question:** do engineered and train-selected non-leaky features improve CRMLS SingleFamilyResidence `ClosePrice` prediction enough to support better listing-price review, valuation triage, and manual-review targeting?

**Core rule:** train creates candidate pipelines, validation chooses among completed candidate pipelines, and June test evaluates the locked winner once.

## 1. Setup And Inputs

This notebook uses the June-refreshed Week 3 cleaned dataset and the Week 5 model results as the benchmark.

Aidan's Week 6 requirements reflected here:
- add interpretable feature engineering
- carry forward the school district layer
- retrain models with the updated feature set
- compare old vs new feature sets
- explain model and feature-selection strengths/weaknesses

Slack channel search was not discoverable in this environment, so the current thread and Week 6 screenshot are treated as the requirement source.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import Markdown
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "outputs" / "week3_preprocessing" / "crmls_sfr_quality_cleaned_202501_202606.csv"
WEEK5_METRICS_PATH = ROOT / "outputs" / "week5_model_comparison" / "week5_final_test_metrics.csv"
WEEK5_SEGMENT_PATH = ROOT / "outputs" / "week5_model_comparison" / "week5_segment_errors.csv"
WEEK5_TEST_PREDICTIONS_PATH = ROOT / "outputs" / "week5_model_comparison" / "week5_selected_model_test_predictions.csv"
OUTPUT_DIR = ROOT / "outputs" / "week6_feature_engineering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from week6_model_pipeline import CRMLSFeaturePreprocessor, ColumnSelector, add_engineered_features

TARGET = "ClosePrice"
RANDOM_STATE = 42
VALIDATION_MONTH = "2026-05"
TEST_MONTH = "2026-06"
TRAIN_START_MONTH = "2025-02"
TRAIN_END_MONTH = "2026-04"
MISSINGNESS_LIMIT = 0.90

print("Data:", DATA_PATH)
print("Output:", OUTPUT_DIR)
print("Train:", TRAIN_START_MONTH, "to", TRAIN_END_MONTH, "| Validation:", VALIDATION_MONTH, "| Test:", TEST_MONTH)

Data: /Users/amyliu/Desktop/summer intern/outputs/week3_preprocessing/crmls_sfr_quality_cleaned_202501_202606.csv
Output: /Users/amyliu/Desktop/summer intern/outputs/week6_feature_engineering
Train: 2025-02 to 2026-04 | Validation: 2026-05 | Test: 2026-06


## 2. Load Data And Confirm Scope

Only `Residential` / `SingleFamilyResidence` rows are eligible. Rows are not dropped blindly; the filters below are the explicit modeling population and target-date requirements.

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)

df = pd.read_csv(DATA_PATH, low_memory=False)
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df["close_month"] = pd.PeriodIndex(df["close_month"].astype(str), freq="M").astype(str)

df = df[
    df["PropertyType"].astype(str).str.strip().eq("Residential")
    & df["PropertySubType"].astype(str).str.strip().eq("SingleFamilyResidence")
    & df["CloseDate"].notna()
].copy()

school_rows = int(df["UnifiedSchoolDistrictName"].notna().sum())
print("Rows:", len(df))
print("Months:", df["close_month"].min(), "to", df["close_month"].max())
print("Rows with ClosePrice available:", int(df[TARGET].notna().sum()))
print("June max CloseDate:", df.loc[df["close_month"].eq(TEST_MONTH), "CloseDate"].max())
print("Unified school district matched rows:", school_rows)

Rows: 187750
Months: 2025-01 to 2026-06
Rows with ClosePrice available: 187750
June max CloseDate: 2026-06-30 00:00:00
Unified school district matched rows: 142197


## 3. Feature Engineering Rationale

These features are allowed because they describe property structure, carrying cost, or geographic market segmentation before model scoring.

| Feature | Why it helps pricing decisions |
| --- | --- |
| `log_living_area` | handles diminishing price effect of additional square footage |
| `log_lot_size_sqft` | reduces skew in lot-size effects |
| `bed_bath_ratio`, `bath_per_bedroom`, `total_bed_bath` | captures layout functionality beyond raw bed/bath counts |
| `garage_present`, `garage_per_bedroom` | captures parking utility, important in many California markets |
| `fireplaces_per_bedroom` | captures amenity intensity without over-weighting large homes |
| `association_fee_present` | separates HOA/fee properties from no-fee properties |
| `tax_per_living_sqft` | proxy for carrying cost and assessed-value intensity |
| school district frequency encoding | train-only geographic market-size signal produced by the fitted preprocessor |
| school district density fields | geographic segmentation, not causal school-quality claims |

In [3]:
df = add_engineered_features(df)

engineered_features = [
    "log_living_area", "log_lot_size_sqft", "bed_bath_ratio", "bath_per_bedroom",
    "total_bed_bath", "garage_present", "garage_per_bedroom", "fireplaces_per_bedroom",
    "association_fee_present", "tax_per_living_sqft",
]

missing_engineered = [col for col in engineered_features if col not in df.columns]
if missing_engineered:
    raise ValueError(f"Missing engineered features: {missing_engineered}")

print("Engineered feature count:", len(engineered_features))

Engineered feature count: 10


## 4. Leakage Guardrail

Leakage fields are blocked before any candidate pipeline is trained. This prevents the feature-selection methods from accidentally learning sale-process or target-derived information.

In [4]:
FORBIDDEN_FEATURES = {
    TARGET,
    "ListPrice",
    "OriginalListPrice",
    "ClosePrice_to_ListPrice_ratio",
    "price_per_sqft_audit",
    "DaysOnMarket",
    "MlsStatus",
    "ContractStatusChangeDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",
    "UnparsedAddress",
    "ListAgentEmail",
    "ListAgentAOR",
    "ListOfficeName",
    "BuyerOfficeName",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "BuyerAgentMlsId",
    "source_month",
    "split",
}

print("Defined leakage blocklist:", len(FORBIDDEN_FEATURES), "fields.")


Defined leakage blocklist: 23 fields.


In [5]:
X5_NUMERIC = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet",
    "LotSizeAcres", "LotSizeArea", "YearBuilt", "property_age_at_close",
    "bed_bath_ratio", "living_area_to_lot_ratio", "Latitude", "Longitude",
    "GarageSpaces", "ParkingTotal", "AssociationFee", "Stories", "MainLevelBedrooms",
    "UnifiedSchoolDistrictEnrollTotal", "UnifiedSchoolDistrictAreaSqMi",
    "UnifiedSchoolDistrictEnrollmentDensity", "close_month_sin", "close_month_cos",
]

X5_CATEGORICAL = [
    "City", "PostalCode", "CountyOrParish", "MLSAreaMajor", "HighSchoolDistrict",
    "UnifiedSchoolDistrictName", "UnifiedSchoolDistrictCounty",
    "UnifiedSchoolDistrictLocaleDesc", "Levels", "AssociationFeeFrequency", "StateOrProvince",
]

X5_BINARY = [
    "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ViewYN_was_missing", "PoolPrivateYN_was_missing", "AttachedGarageYN_was_missing",
    "FireplaceYN_was_missing", "NewConstructionYN_was_missing",
    "flag_unified_school_district_missing",
]

print("Defined X5 baseline raw feature lists.")


Defined X5 baseline raw feature lists.


In [6]:
SCHOOL_NUMERIC = [
    "UnifiedSchoolDistrictEnrollTotal",
    "UnifiedSchoolDistrictAreaSqMi",
    "UnifiedSchoolDistrictEnrollmentDensity",
]

SCHOOL_CATEGORICAL = [
    "HighSchoolDistrict",
    "UnifiedSchoolDistrictName",
    "UnifiedSchoolDistrictCounty",
    "UnifiedSchoolDistrictLocaleDesc",
]

SCHOOL_BINARY = ["flag_unified_school_district_missing"]

X6_EXTRA_NUMERIC = [
    "log_living_area", "log_lot_size_sqft", "bath_per_bedroom", "total_bed_bath",
    "garage_per_bedroom", "fireplaces_per_bedroom", "tax_per_living_sqft",
]

X6_EXTRA_BINARY = ["garage_present", "association_fee_present"]

X5_NUMERIC_NO_SCHOOL = [col for col in X5_NUMERIC if col not in SCHOOL_NUMERIC]
X5_CATEGORICAL_NO_SCHOOL = [col for col in X5_CATEGORICAL if col not in SCHOOL_CATEGORICAL]
X5_BINARY_NO_SCHOOL = [col for col in X5_BINARY if col not in SCHOOL_BINARY]
X6_EXTRA_NUMERIC_NO_SCHOOL = list(X6_EXTRA_NUMERIC)

FEATURE_SPECS = {
    "X5_no_school": {
        "numeric": X5_NUMERIC_NO_SCHOOL,
        "categorical": X5_CATEGORICAL_NO_SCHOOL,
        "binary": X5_BINARY_NO_SCHOOL,
        "description": "Week 5 fixed feature set with school-district fields removed.",
    },
    "X5_fixed": {
        "numeric": X5_NUMERIC,
        "categorical": X5_CATEGORICAL,
        "binary": X5_BINARY,
        "description": "Week 5 fixed non-leaky feature set.",
    },
    "X6_engineered_no_school": {
        "numeric": list(dict.fromkeys(X5_NUMERIC_NO_SCHOOL + X6_EXTRA_NUMERIC_NO_SCHOOL)),
        "categorical": X5_CATEGORICAL_NO_SCHOOL,
        "binary": list(dict.fromkeys(X5_BINARY_NO_SCHOOL + X6_EXTRA_BINARY)),
        "description": "Week 6 engineered feature pool with school-district fields removed.",
    },
    "X6_engineered": {
        "numeric": list(dict.fromkeys(X5_NUMERIC + X6_EXTRA_NUMERIC)),
        "categorical": X5_CATEGORICAL,
        "binary": list(dict.fromkeys(X5_BINARY + X6_EXTRA_BINARY)),
        "description": "Week 6 expanded non-leaky feature pool with train-only district frequency.",
    },
}

for spec_name, spec in FEATURE_SPECS.items():
    raw_features = set(spec["numeric"] + spec["categorical"] + spec["binary"])
    leakage = sorted(raw_features.intersection(FORBIDDEN_FEATURES))
    if leakage:
        raise ValueError(f"{spec_name} contains leakage features: {leakage}")

print("Leakage guardrail passed for feature specs:", list(FEATURE_SPECS))

Leakage guardrail passed for feature specs: ['X5_no_school', 'X5_fixed', 'X6_engineered_no_school', 'X6_engineered']


## 5. Chronological Split

Train builds candidate pipelines. Validation chooses among completed pipelines. June test target filtering, preprocessing, and scoring are delayed until the winning pipeline is locked.

January 2025 is excluded from training to stay aligned with the Week 5 benchmark window. This makes Week 5 versus Week 6 comparisons methodologically comparable.

In [7]:
month = pd.PeriodIndex(df["close_month"].astype(str), freq="M")
train_raw = df[(month >= pd.Period(TRAIN_START_MONTH, freq="M")) & (month <= pd.Period(TRAIN_END_MONTH, freq="M"))].copy()
validation_raw = df[month == pd.Period(VALIDATION_MONTH, freq="M")].copy()
test_raw = df[month == pd.Period(TEST_MONTH, freq="M")].copy()

train = train_raw[train_raw[TARGET].notna()].sort_values("CloseDate").reset_index(drop=True).copy()
validation = validation_raw[validation_raw[TARGET].notna()].sort_values("CloseDate").reset_index(drop=True).copy()

if min(len(train), len(validation), len(test_raw)) == 0:
    raise ValueError("Train, validation, or June test partition is empty.")

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows_before_filter": [len(train_raw), len(validation_raw), len(test_raw)],
    "rows_with_closeprice": [len(train), len(validation), int(test_raw[TARGET].notna().sum())],
    "start_month": [train["close_month"].min(), validation["close_month"].min(), test_raw["close_month"].min()],
    "end_month": [train["close_month"].max(), validation["close_month"].max(), test_raw["close_month"].max()],
})

split_summary

,split,rows_before_filter,rows_with_closeprice,start_month,end_month
0,train,154769,154769,2025-02,2026-04
1,validation,12002,12002,2026-05,2026-05
2,test,12851,12851,2026-06,2026-06


## 6. Train-Fitted Training Filter

Extreme target and price-per-square-foot thresholds are learned from train and used only to stabilize model fitting. Primary validation and June test metrics use the full eligible transaction populations, because filtering evaluation rows by actual sale price would make the reported error too optimistic.

In [8]:
def fit_training_filter(train_frame):
    price_low, price_high = train_frame[TARGET].quantile([0.005, 0.995])

    train_filtered = train_frame[train_frame[TARGET].between(price_low, price_high)].copy()

    ppsf_low = np.nan
    ppsf_high = np.nan
    if "price_per_sqft_audit" in train_filtered.columns:
        train_ppsf = pd.to_numeric(train_filtered["price_per_sqft_audit"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        train_ppsf = train_ppsf.dropna()
        if len(train_ppsf):
            ppsf_low, ppsf_high = train_ppsf.quantile([0.005, 0.995])
            values = pd.to_numeric(train_filtered["price_per_sqft_audit"], errors="coerce").replace([np.inf, -np.inf], np.nan)
            train_filtered = train_filtered[values.isna() | values.between(ppsf_low, ppsf_high)].copy()

    return train_filtered.reset_index(drop=True), {
        "closeprice_p005_train": float(price_low),
        "closeprice_p995_train": float(price_high),
        "price_per_sqft_p005_train": float(ppsf_low) if pd.notna(ppsf_low) else np.nan,
        "price_per_sqft_p995_train": float(ppsf_high) if pd.notna(ppsf_high) else np.nan,
    }


train, outlier_params = fit_training_filter(train)

split_summary["rows_after_training_filter"] = [len(train), len(validation), np.nan]
print("Train-fitted ClosePrice filter:", round(outlier_params["closeprice_p005_train"], 0), "to", round(outlier_params["closeprice_p995_train"], 0))
print("Train-fitted price/sqft filter:", round(outlier_params["price_per_sqft_p005_train"], 2), "to", round(outlier_params["price_per_sqft_p995_train"], 2))
split_summary

Train-fitted ClosePrice filter: 188420.0 to 8554609.0
Train-fitted price/sqft filter: 162.5 to 2092.38


,split,rows_before_filter,rows_with_closeprice,start_month,end_month,rows_after_training_filter
0,train,154769,154769,2025-02,2026-04,151691.0
1,validation,12002,12002,2026-05,2026-05,12002.0
2,test,12851,12851,2026-06,2026-06,NaN


## 7. Metrics And Preprocessing

The preprocessor is fit inside each candidate pipeline using train only. Validation and test receive frozen train-fitted transformations.

In [9]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    keep = y_true != 0
    return float(np.mean(np.abs((y_true[keep] - y_pred[keep]) / y_true[keep])))


def mdape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    keep = y_true != 0
    return float(np.median(np.abs((y_true[keep] - y_pred[keep]) / y_true[keep])))


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def evaluate_predictions(y_true, y_pred, prefix):
    return {
        f"{prefix}_R2": float(r2_score(y_true, y_pred)),
        f"{prefix}_MAE": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}_RMSE": rmse(y_true, y_pred),
        f"{prefix}_MAPE": mape(y_true, y_pred),
        f"{prefix}_MdAPE": mdape(y_true, y_pred),
    }


def build_train_validation_matrices(train_frame, validation_frame, spec):
    preprocessor = CRMLSFeaturePreprocessor(spec, missingness_limit=MISSINGNESS_LIMIT)
    x_train = preprocessor.fit_transform(train_frame)
    x_validation = preprocessor.transform(validation_frame)

    if x_train.shape[1] == 0:
        raise ValueError("No features available after preprocessing.")

    return x_train, x_validation, preprocessor, preprocessor.get_preprocess_params()


print("Defined metrics and train-fitted raw-data preprocessor.")

Defined metrics and train-fitted raw-data preprocessor.


## 8. Train-Only Feature Selectors

These selectors are fit on `x_train` only. They create locked candidate pipelines. Validation does not select raw variables.

In [10]:
def correlation_filter_features(x_train, threshold=0.95):
    corr = x_train.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    dropped = [col for col in upper.columns if any(upper[col] > threshold)]
    selected = [col for col in x_train.columns if col not in dropped]
    return selected, dropped


def lasso_select_features(x_train, y_train, random_state=RANDOM_STATE):
    y_values = np.asarray(y_train, dtype=float)
    y_scaled = (y_values - y_values.mean()) / (y_values.std() or 1.0)
    model = LassoCV(
        alphas=np.logspace(-3, 3, 30),
        cv=TimeSeriesSplit(n_splits=3),
        max_iter=5000,
        n_jobs=-1,
        random_state=random_state,
    )
    model.fit(x_train, y_scaled)
    coefficient_abs = pd.Series(np.abs(model.coef_), index=x_train.columns)
    selected = coefficient_abs[coefficient_abs > 1e-8].sort_values(ascending=False).index.tolist()
    if len(selected) < 10:
        selected = coefficient_abs.sort_values(ascending=False).head(min(30, len(coefficient_abs))).index.tolist()
    dropped = [col for col in x_train.columns if col not in selected]
    return selected, dropped, {"lasso_alpha": float(model.alpha_)}


def rf_importance_select_features(x_train, y_train, top_n=35, random_state=RANDOM_STATE):
    model = RandomForestRegressor(
        n_estimators=40,
        max_depth=18,
        min_samples_leaf=20,
        max_features=0.6,
        n_jobs=-1,
        random_state=random_state,
    )
    model.fit(x_train, y_train)
    importances = pd.Series(model.feature_importances_, index=x_train.columns).sort_values(ascending=False)
    selected = importances.head(min(top_n, len(importances))).index.tolist()
    dropped = [col for col in x_train.columns if col not in selected]
    return selected, dropped, {"rf_selector_top_importance": importances.head(15).to_dict()}


print("Defined train-only feature selectors with time-aware LassoCV.")

Defined train-only feature selectors with time-aware LassoCV.


## 9. Candidate Pipeline Builder

Each candidate below is a complete train-fitted pipeline: preprocessing, optional feature selection, and model training all happen before validation scoring.

In [11]:
def make_estimator(model_type):
    if model_type == "LinearRegression":
        return LinearRegression()
    if model_type == "RidgeCV":
        return TransformedTargetRegressor(
            regressor=RidgeCV(alphas=np.logspace(-2, 6, 25), cv=TimeSeriesSplit(n_splits=3)),
            transformer=StandardScaler(),
        )
    if model_type == "LassoCV":
        return TransformedTargetRegressor(
            regressor=LassoCV(
                alphas=np.logspace(-3, 3, 30),
                cv=TimeSeriesSplit(n_splits=3),
                max_iter=5000,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
            transformer=StandardScaler(),
        )
    if model_type == "DecisionTree":
        return DecisionTreeRegressor(max_depth=20, min_samples_leaf=50, random_state=RANDOM_STATE)
    if model_type == "RandomForest":
        return RandomForestRegressor(
            n_estimators=50,
            max_depth=22,
            min_samples_leaf=10,
            max_features=0.6,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    raise ValueError(f"Unknown model type: {model_type}")

print("Defined model estimators.")


Defined model estimators.


In [12]:
CANDIDATE_PIPELINES = [
    {"pipeline": "X5_no_school + Random Forest", "feature_spec": "X5_no_school", "selector": "none", "model_type": "RandomForest"},
    {"pipeline": "X5_fixed + Linear Regression", "feature_spec": "X5_fixed", "selector": "none", "model_type": "LinearRegression"},
    {"pipeline": "X5_fixed + Ridge", "feature_spec": "X5_fixed", "selector": "none", "model_type": "RidgeCV"},
    {"pipeline": "X5_fixed + Lasso", "feature_spec": "X5_fixed", "selector": "none", "model_type": "LassoCV"},
    {"pipeline": "X5_fixed + Decision Tree", "feature_spec": "X5_fixed", "selector": "none", "model_type": "DecisionTree"},
    {"pipeline": "X5_fixed + Random Forest", "feature_spec": "X5_fixed", "selector": "none", "model_type": "RandomForest"},
    {"pipeline": "X6_no_school + Random Forest", "feature_spec": "X6_engineered_no_school", "selector": "none", "model_type": "RandomForest"},
    {"pipeline": "X6_full + Random Forest", "feature_spec": "X6_engineered", "selector": "none", "model_type": "RandomForest"},
    {"pipeline": "Correlation-filtered X6 + Ridge", "feature_spec": "X6_engineered", "selector": "correlation_filter", "model_type": "RidgeCV"},
    {"pipeline": "Lasso-selected X6 + Linear Regression", "feature_spec": "X6_engineered", "selector": "lasso_selection", "model_type": "LinearRegression"},
    {"pipeline": "Lasso-selected X6 + Ridge", "feature_spec": "X6_engineered", "selector": "lasso_selection", "model_type": "RidgeCV"},
    {"pipeline": "RF-importance-selected X6 + Random Forest", "feature_spec": "X6_engineered", "selector": "rf_importance", "model_type": "RandomForest"},
    {"pipeline": "Combined-selected X6 + Random Forest", "feature_spec": "X6_engineered", "selector": "combined_lasso_rf", "model_type": "RandomForest"},
]

print("Defined candidate pipeline list:", len(CANDIDATE_PIPELINES), "pipelines.")


Defined candidate pipeline list: 13 pipelines.


In [13]:
def select_features_for_candidate(candidate, x_train_full, y_train):
    selector = candidate["selector"]
    selector_params = {}

    if selector == "none":
        selected_features = x_train_full.columns.tolist()
        removed_features = []
    elif selector == "correlation_filter":
        selected_features, removed_features = correlation_filter_features(x_train_full)
    elif selector == "lasso_selection":
        selected_features, removed_features, selector_params = lasso_select_features(x_train_full, y_train)
    elif selector == "rf_importance":
        selected_features, removed_features, selector_params = rf_importance_select_features(x_train_full, y_train)
    elif selector == "combined_lasso_rf":
        lasso_features, _, lasso_params = lasso_select_features(x_train_full, y_train)
        rf_features, _, rf_params = rf_importance_select_features(x_train_full, y_train)
        selected_features = sorted(set(lasso_features).union(rf_features))
        removed_features = [col for col in x_train_full.columns if col not in selected_features]
        selector_params = {
            "lasso_alpha": lasso_params.get("lasso_alpha"),
            "combined_rule": "union_lasso_and_rf_top_features",
        }
    else:
        raise ValueError(f"Unknown selector: {selector}")

    return selected_features, removed_features, selector_params


print("Defined candidate feature-selection dispatcher.")

Defined candidate feature-selection dispatcher.


In [14]:
def package_raw_prediction_pipeline(preprocessor, selected_features, model, x_train_full):
    selector_step = ColumnSelector(selected_features).fit(x_train_full)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("selector", selector_step),
        ("model", model),
    ])


def build_candidate_result(candidate, spec, train_frame, validation_frame, selected_features, removed_features, y_train, y_validation, train_pred, validation_pred, x_train_full):
    result = {
        "pipeline": candidate["pipeline"],
        "feature_spec": candidate["feature_spec"],
        "selector": candidate["selector"],
        "model_type": candidate["model_type"],
        "raw_feature_count": len(spec["numeric"] + spec["categorical"] + spec["binary"]),
        "transformed_feature_count_before_selection": int(x_train_full.shape[1]),
        "selected_feature_count": len(selected_features),
        "removed_feature_count": len(removed_features),
        "train_rows": len(train_frame),
        "validation_rows": len(validation_frame),
        **evaluate_predictions(y_train, train_pred, "train"),
        **evaluate_predictions(y_validation, validation_pred, "validation"),
    }
    result["train_minus_validation_R2"] = result["train_R2"] - result["validation_R2"]
    result["validation_minus_train_MdAPE"] = result["validation_MdAPE"] - result["train_MdAPE"]
    return result


print("Defined candidate packaging and metric helpers.")

Defined candidate packaging and metric helpers.


In [15]:
def fit_candidate_pipeline(candidate, train_frame, validation_frame):
    spec = FEATURE_SPECS[candidate["feature_spec"]]
    x_train_full, x_validation_full, preprocessor, preprocess_params = build_train_validation_matrices(
        train_frame,
        validation_frame,
        spec,
    )
    y_train = train_frame[TARGET].astype(float)
    y_validation = validation_frame[TARGET].astype(float)

    selected_features, removed_features, selector_params = select_features_for_candidate(
        candidate,
        x_train_full,
        y_train,
    )
    x_train = x_train_full[selected_features]
    x_validation = x_validation_full[selected_features]

    model = make_estimator(candidate["model_type"])
    model.fit(x_train, y_train)
    train_pred = model.predict(x_train)
    validation_pred = model.predict(x_validation)

    result = build_candidate_result(
        candidate,
        spec,
        train_frame,
        validation_frame,
        selected_features,
        removed_features,
        y_train,
        y_validation,
        train_pred,
        validation_pred,
        x_train_full,
    )

    fitted = {
        "candidate": candidate,
        "model": model,
        "raw_prediction_pipeline": package_raw_prediction_pipeline(preprocessor, selected_features, model, x_train_full),
        "preprocessor": preprocessor,
        "selected_features": selected_features,
        "removed_features": removed_features,
        "preprocess_params": preprocess_params,
        "selector_params": selector_params,
        "x_train": x_train,
        "x_validation": x_validation,
        "y_train": y_train,
        "y_validation": y_validation,
        "train_frame": train_frame,
        "validation_frame": validation_frame,
        "result": result,
    }
    return result, fitted


print("Defined candidate pipeline builder:", len(CANDIDATE_PIPELINES), "pipelines.")

Defined candidate pipeline builder: 13 pipelines.


## 10. Train Candidate Pipelines

This is the only phase where feature selection is fitted. The validation set is not used to create or modify feature lists.

In [16]:
fitted_pipelines = {}
candidate_results = []

for candidate in CANDIDATE_PIPELINES:
    print("Training candidate pipeline:", candidate["pipeline"])
    result, fitted = fit_candidate_pipeline(candidate, train, validation)
    candidate_results.append(result)
    fitted_pipelines[candidate["pipeline"]] = fitted

validation_results = pd.DataFrame(candidate_results).sort_values(
    ["validation_MdAPE", "validation_MAPE", "validation_R2"],
    ascending=[True, True, False],
).reset_index(drop=True)

validation_results.to_csv(OUTPUT_DIR / "week6_candidate_pipeline_validation_metrics.csv", index=False)
validation_results[[
    "pipeline", "selector", "model_type", "selected_feature_count",
    "train_R2", "validation_R2", "validation_MAPE", "validation_MdAPE",
    "train_minus_validation_R2", "validation_minus_train_MdAPE",
]]

Training candidate pipeline: X5_no_school + Random Forest


Training candidate pipeline: X5_fixed + Linear Regression


Training candidate pipeline: X5_fixed + Ridge


Training candidate pipeline: X5_fixed + Lasso


Training candidate pipeline: X5_fixed + Decision Tree


...

Training candidate pipeline: Lasso-selected X6 + Linear Regression


Training candidate pipeline: Lasso-selected X6 + Ridge


Training candidate pipeline: RF-importance-selected X6 + Random Forest


Training candidate pipeline: Combined-selected X6 + Random Forest


,pipeline,selector,model_type,selected_feature_count,train_R2,validation_R2,validation_MAPE,validation_MdAPE,train_minus_validation_R2,validation_minus_train_MdAPE
0,X5_fixed + Random Forest,none,RandomForest,77,0.921254,0.678848,0.147429,0.084170,0.242405,0.018539
1,X6_full + Random Forest,none,RandomForest,91,0.920472,0.675134,0.149258,0.084426,0.245339,0.018417
2,RF-importance-selected X6 + Random Forest,rf_importance,RandomForest,35,0.920102,0.676580,0.148093,0.084981,0.243522,0.018517
3,Combined-selected X6 + Random Forest,combined_lasso_rf,RandomForest,54,0.921443,0.680202,0.148388,0.084983,0.241241,0.019010
4,X6_no_school + Random Forest,none,RandomForest,74,0.917601,0.674572,0.151372,0.086585,0.243030,0.019806
5,X5_no_school + Random Forest,none,RandomForest,62,0.917316,0.676855,0.151525,0.086987,0.240460,0.020074
6,X5_fixed + Decision Tree,none,DecisionTree,77,0.835586,0.623285,0.187393,0.111219,0.212301,0.014773
7,X5_fixed + Ridge,none,RidgeCV,77,0.629389,0.543511,0.382656,0.260402,0.085878,0.006387
8,X5_fixed + Linear Regression,none,LinearRegression,77,0.629408,0.543655,0.383039,0.261002,0.085752,0.006686
9,Correlation-filtered X6 + Ridge,correlation_filter,RidgeCV,72,0.630255,0.548359,0.381712,0.262273,0.081896,0.008619


### Candidate Training Result Summary

- Random Forest pipelines dominate validation performance.
- `X5_fixed + Random Forest` ranks first, so the Week 5 model structure remains the strongest candidate.
- `X6_full + Random Forest` does not improve validation MdAPE, so added engineered features do not show clear incremental value before selection.
- X6 feature selection reduces feature count, but does not materially improve validation error.
- No-school pipelines perform worse, which supports keeping the school-district layer.
- Linear and regularized linear models underperform, suggesting the pricing pattern is strongly nonlinear.

## 11. Validation Chooses Among Completed Pipelines

Use the May 2026 validation set to compare **pre-defined completed modeling pipelines**. Validation does not refit feature selectors and does not manually change feature lists.

| Pipeline group | Feature set or selection strategy | Model candidates |
| --- | --- | --- |
| School-layer control | X5 without school-district fields | Random Forest |
| Baseline | `X5_full_non_leaky` fixed features, including school layer | Linear Regression, Ridge, Lasso, Decision Tree, Random Forest |
| Feature-engineering control | X6 full engineered features without selector | Random Forest |
| School-layer + engineering control | X6 engineered features without school layer | Random Forest |
| Pipeline A | Correlation-filtered X6 features fitted on train | Ridge |
| Pipeline B | Lasso-selected X6 features fitted on train | Linear Regression, Ridge |
| Pipeline C | RF-importance-selected X6 features fitted on train | Random Forest |
| Pipeline D | Combined Lasso + RF selected X6 features fitted on train | Random Forest |

Validation is used to choose:
- best completed pipeline
- best model family
- best hyperparameter setting already encoded in that pipeline

Selection rule: lowest validation `MdAPE`, then `MAPE`, then `R2`.

In [17]:
selected_pipeline_name = validation_results.loc[0, "pipeline"]
selected_fit = fitted_pipelines[selected_pipeline_name]

print("Selected pipeline:", selected_pipeline_name)
print("Selector:", selected_fit["candidate"]["selector"])
print("Model:", selected_fit["candidate"]["model_type"])
print("Selected features:", len(selected_fit["selected_features"]))
print("Removed features:", len(selected_fit["removed_features"]))
print("Validation R2:", round(validation_results.loc[0, "validation_R2"], 4))
print("Validation MAPE:", round(validation_results.loc[0, "validation_MAPE"], 4))
print("Validation MdAPE:", round(validation_results.loc[0, "validation_MdAPE"], 4))

validation_results.head(5)

Selected pipeline: X5_fixed + Random Forest
Selector: none
Model: RandomForest
Selected features: 77
Removed features: 0
Validation R2: 0.6788
Validation MAPE: 0.1474
Validation MdAPE: 0.0842


,pipeline,feature_spec,selector,model_type,raw_feature_count,transformed_feature_count_before_selection,selected_feature_count,removed_feature_count,train_rows,validation_rows,...,train_RMSE,train_MAPE,train_MdAPE,validation_R2,validation_MAE,validation_RMSE,validation_MAPE,validation_MdAPE,train_minus_validation_R2,validation_minus_train_MdAPE
0,X5_fixed + Random Forest,X5_fixed,none,RandomForest,44,77,77,0,151691,12002,...,260907.776515,0.102383,0.065632,0.678848,218759.550407,769945.640544,0.147429,0.084170,0.242405,0.018539
1,X6_full + Random Forest,X6_engineered,none,RandomForest,54,91,91,0,151691,12002,...,262198.574217,0.103101,0.066009,0.675134,220708.896663,774385.406120,0.149258,0.084426,0.245339,0.018417
2,RF-importance-selected X6 + Random Forest,X6_engineered,rf_importance,RandomForest,54,91,35,56,151691,12002,...,262808.690780,0.103478,0.066465,0.676580,220198.358832,772659.598521,0.148093,0.084981,0.243522,0.018517
3,Combined-selected X6 + Random Forest,X6_engineered,combined_lasso_rf,RandomForest,54,91,54,37,151691,12002,...,260593.783708,0.102638,0.065973,0.680202,219311.594809,768320.596439,0.148388,0.084983,0.241241,0.019010
4,X6_no_school + Random Forest,X6_engineered_no_school,none,RandomForest,45,74,74,0,151691,12002,...,266889.690357,0.104939,0.066780,0.674572,224184.849291,775055.017771,0.151372,0.086585,0.243030,0.019806


- The selection rule prioritizes lowest validation MdAPE, then MAPE, then R2.
- `X5_fixed + Random Forest` is selected because it has the best validation MdAPE.
- This means the Week 6 selected alternatives were tested, but did not justify replacing the Week 5 RF structure.


## 12. Controlled Feature-Set Comparison On Validation

This section explains why the validation stage selected `X5_fixed + Random Forest`.

Each comparison changes only one design choice at a time:

- **School layer:** compare the same RF model with and without school-district fields.
- **Engineered features:** compare X5 RF against X6 full RF before feature selection.
- **Feature selection:** compare X6 full RF against selected-X6 RF.

All comparisons use May validation only. June is not used here.

In [18]:
def get_validation_row(pipeline_name):
    rows = validation_results[validation_results["pipeline"].eq(pipeline_name)]
    if len(rows) == 0:
        return None
    return rows.iloc[0]


selected_x6_rf = (
    validation_results[
        validation_results["feature_spec"].eq("X6_engineered")
        & validation_results["model_type"].eq("RandomForest")
        & validation_results["selector"].ne("none")
    ]
    .sort_values(["validation_MdAPE", "validation_MAPE", "validation_R2"], ascending=[True, True, False])
)
selected_x6_rf_name = selected_x6_rf.iloc[0]["pipeline"] if len(selected_x6_rf) else None

comparison_pairs = [
    {
        "comparison": "School layer effect within X5 RF",
        "baseline_pipeline": "X5_no_school + Random Forest",
        "challenger_pipeline": "X5_fixed + Random Forest",
        "question_answered": "Does adding the school-district layer improve the Week 5 RF feature set?",
    },
    {
        "comparison": "School layer effect within X6 RF",
        "baseline_pipeline": "X6_no_school + Random Forest",
        "challenger_pipeline": "X6_full + Random Forest",
        "question_answered": "Does adding the school-district layer improve the engineered RF feature set?",
    },
    {
        "comparison": "Engineered feature effect without selector",
        "baseline_pipeline": "X5_fixed + Random Forest",
        "challenger_pipeline": "X6_full + Random Forest",
        "question_answered": "Do the engineered features improve RF before any feature selector removes variables?",
    },
    {
        "comparison": "Selector effect on X6 RF",
        "baseline_pipeline": "X6_full + Random Forest",
        "challenger_pipeline": selected_x6_rf_name,
        "question_answered": "Does train-only feature selection improve the engineered RF feature pool?",
    },
]

print("Defined controlled comparison pairs.")


Defined controlled comparison pairs.


In [19]:
controlled_rows = []
for item in comparison_pairs:
    baseline = get_validation_row(item["baseline_pipeline"])
    challenger = get_validation_row(item["challenger_pipeline"]) if item["challenger_pipeline"] else None
    if baseline is None or challenger is None:
        continue

    mdape_delta = challenger["validation_MdAPE"] - baseline["validation_MdAPE"]
    mape_delta = challenger["validation_MAPE"] - baseline["validation_MAPE"]
    r2_delta = challenger["validation_R2"] - baseline["validation_R2"]
    if mdape_delta < -0.001:
        decision = "improved validation MdAPE"
    elif mdape_delta > 0.001:
        decision = "worse validation MdAPE"
    else:
        decision = "no material MdAPE change"

    controlled_rows.append({
        **item,
        "baseline_validation_R2": baseline["validation_R2"],
        "challenger_validation_R2": challenger["validation_R2"],
        "delta_R2": r2_delta,
        "baseline_validation_MAPE": baseline["validation_MAPE"],
        "challenger_validation_MAPE": challenger["validation_MAPE"],
        "delta_MAPE": mape_delta,
        "baseline_validation_MdAPE": baseline["validation_MdAPE"],
        "challenger_validation_MdAPE": challenger["validation_MdAPE"],
        "delta_MdAPE": mdape_delta,
        "validation_decision": decision,
    })

controlled_feature_set_comparison = pd.DataFrame(controlled_rows)
controlled_feature_set_comparison.to_csv(
    OUTPUT_DIR / "week6_controlled_feature_set_comparison.csv",
    index=False,
)

controlled_feature_set_comparison[[
    "comparison",
    "baseline_pipeline",
    "challenger_pipeline",
    "delta_R2",
    "delta_MAPE",
    "delta_MdAPE",
    "validation_decision",
]]

,comparison,baseline_pipeline,challenger_pipeline,delta_R2,delta_MAPE,delta_MdAPE,validation_decision
0,School layer effect within X5 RF,X5_no_school + Random Forest,X5_fixed + Random Forest,0.001993,-0.004096,-0.002816,improved validation MdAPE
1,School layer effect within X6 RF,X6_no_school + Random Forest,X6_full + Random Forest,0.000562,-0.002114,-0.002160,improved validation MdAPE
2,Engineered feature effect without selector,X5_fixed + Random Forest,X6_full + Random Forest,-0.003714,0.001829,0.000256,no material MdAPE change
3,Selector effect on X6 RF,X6_full + Random Forest,RF-importance-selected X6 + Random Forest,0.001446,-0.001165,0.000555,no material MdAPE change


### Controlled Comparison Interpretation

- **School layer:** improves validation performance in both X5 and X6, so school-district features should be kept.
- **Engineered features:** X6 full does not outperform X5 full, so the added engineered features do not justify replacing X5.
- **Feature selection:** RF selection reduces feature count, but does not improve validation MdAPE, so the selected-X6 pipeline is not stronger than X6 full.
- **Decision:** keep `X5_fixed + Random Forest` as the locked Week 6 model.

## 13. Final June Test Once

The selected pipeline is locked before this cell. June is a repeated comparison month for consistency with Week 5, not a fully untouched final holdout for the whole project. Primary metrics use all eligible June SFR transactions with available `ClosePrice`.

In [20]:
test = test_raw[test_raw[TARGET].notna()].sort_values("CloseDate").reset_index(drop=True).copy()
if len(test) == 0:
    raise ValueError("June test set is empty.")

model = selected_fit["model"]
raw_prediction_pipeline = selected_fit["raw_prediction_pipeline"]
y_train = selected_fit["y_train"]
y_validation = selected_fit["y_validation"]
y_test = test[TARGET].astype(float)

train_pred = model.predict(selected_fit["x_train"])
validation_pred = model.predict(selected_fit["x_validation"])
test_pred = raw_prediction_pipeline.predict(test)

selected_metrics = {
    "pipeline": selected_pipeline_name,
    "feature_spec": selected_fit["candidate"]["feature_spec"],
    "selector": selected_fit["candidate"]["selector"],
    "model_type": selected_fit["candidate"]["model_type"],
    "selected_feature_count": len(selected_fit["selected_features"]),
    "removed_feature_count": len(selected_fit["removed_features"]),
    "test_rows": len(test),
    **evaluate_predictions(y_train, train_pred, "train"),
    **evaluate_predictions(y_validation, validation_pred, "validation"),
    **evaluate_predictions(y_test, test_pred, "test"),
}
selected_metrics["train_minus_validation_R2"] = selected_metrics["train_R2"] - selected_metrics["validation_R2"]
selected_metrics["validation_minus_test_R2"] = selected_metrics["validation_R2"] - selected_metrics["test_R2"]
selected_metrics["test_minus_validation_MdAPE"] = selected_metrics["test_MdAPE"] - selected_metrics["validation_MdAPE"]

final_test_metrics = pd.DataFrame([selected_metrics])
final_test_metrics.to_csv(OUTPUT_DIR / "week6_final_locked_pipeline_test_metrics.csv", index=False)

def in_training_range_population(frame, params):
    filtered = frame[
        frame[TARGET].between(params["closeprice_p005_train"], params["closeprice_p995_train"])
    ].copy()
    ppsf_low = params.get("price_per_sqft_p005_train", np.nan)
    ppsf_high = params.get("price_per_sqft_p995_train", np.nan)
    if "price_per_sqft_audit" in filtered.columns and pd.notna(ppsf_low) and pd.notna(ppsf_high):
        values = pd.to_numeric(filtered["price_per_sqft_audit"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        filtered = filtered[values.isna() | values.between(ppsf_low, ppsf_high)].copy()
    return filtered


validation_in_range = in_training_range_population(validation, outlier_params)
test_in_range = in_training_range_population(test, outlier_params)
sensitivity_rows = []
for split_name, frame in [("validation_in_training_range", validation_in_range), ("june_in_training_range", test_in_range)]:
    if len(frame):
        prediction = raw_prediction_pipeline.predict(frame)
        sensitivity_rows.append({
            "population": split_name,
            "rows": len(frame),
            **evaluate_predictions(frame[TARGET].astype(float), prediction, "metric"),
        })

secondary_sensitivity_metrics = pd.DataFrame(sensitivity_rows)
secondary_sensitivity_metrics.to_csv(OUTPUT_DIR / "week6_secondary_in_range_metrics.csv", index=False)

display(final_test_metrics)
display(secondary_sensitivity_metrics)

,pipeline,feature_spec,selector,model_type,selected_feature_count,removed_feature_count,test_rows,train_R2,train_MAE,train_RMSE,...,validation_MAPE,validation_MdAPE,test_R2,test_MAE,test_RMSE,test_MAPE,test_MdAPE,train_minus_validation_R2,validation_minus_test_R2,test_minus_validation_MdAPE
0,X5_fixed + Random Forest,X5_fixed,none,RandomForest,77,0,12851,0.921254,130885.228789,260907.776515,...,0.147429,0.08417,0.596833,233402.743558,975738.147659,0.14839,0.084821,0.242405,0.082015,0.000651


,population,rows,metric_R2,metric_MAE,metric_RMSE,metric_MAPE,metric_MdAPE
0,validation_in_training_range,11753,0.876486,169772.531519,331886.114873,0.127816,0.082013
1,june_in_training_range,12566,0.879076,169552.650451,331533.995508,0.128015,0.082500


### June Evaluation Interpretation

- **Primary test:** `X5_fixed + Random Forest` is evaluated on all eligible June SFR sales.
- **Primary result:** MdAPE is `8.48%`, but R2 drops to `0.5968` because extreme June transactions remain included.
- **Secondary check:** in-range June performance is much stronger, with R2 `0.8791` and MdAPE `8.25%`.
- **Interpretation:** the model is reliable for typical in-range homes, but less stable for extreme or unusual June transactions.
- **Decision:** use the model for valuation triage, not automatic final pricing.

## 14. Feature Selection Audit

This audit reports what the locked train-fitted pipeline kept or removed. If the locked pipeline uses the fixed X5 feature set with no selector, the correct interpretation is that validation rejected the feature-selection alternatives.

In [21]:
feature_status = pd.DataFrame({
    "feature": selected_fit["selected_features"] + selected_fit["removed_features"],
    "status": ["selected"] * len(selected_fit["selected_features"]) + ["removed"] * len(selected_fit["removed_features"]),
    "pipeline": selected_pipeline_name,
    "selector": selected_fit["candidate"]["selector"],
})

new_engineered_tokens = [
    "log_living_area", "log_lot_size_sqft", "bath_per_bedroom", "total_bed_bath",
    "garage_present", "garage_per_bedroom", "fireplaces_per_bedroom",
    "association_fee_present", "tax_per_living_sqft",
]
feature_status["is_new_engineered_feature"] = feature_status["feature"].apply(
    lambda value: any(token in value for token in new_engineered_tokens)
)
feature_status["feature_origin"] = np.where(feature_status["is_new_engineered_feature"], "new_engineered", "existing_x5_or_encoded")

feature_status.to_csv(OUTPUT_DIR / "week6_selected_removed_features.csv", index=False)

selected_engineered = feature_status[
    (feature_status["status"] == "selected") & feature_status["is_new_engineered_feature"]
].copy()
removed_features = feature_status[feature_status["status"] == "removed"].copy()

feature_audit_summary = pd.DataFrame([{
    "selected_pipeline": selected_pipeline_name,
    "selector": selected_fit["candidate"]["selector"],
    "selected_features": len(selected_fit["selected_features"]),
    "removed_features": len(selected_fit["removed_features"]),
    "selected_new_engineered_features": len(selected_engineered),
    "removed_new_engineered_features": int(removed_features["is_new_engineered_feature"].sum()) if len(removed_features) else 0,
    "report_interpretation": (
        "No train-fitted feature selector was adopted. Validation selected the fixed X5 benchmark pipeline."
        if selected_fit["candidate"]["selector"] == "none"
        else "The locked pipeline used train-fitted feature selection. Selected and removed features are listed below."
    ),
}])
feature_audit_summary.to_csv(OUTPUT_DIR / "week6_feature_selection_audit_summary.csv", index=False)

display(feature_audit_summary)

if len(selected_engineered):
    print("Selected engineered features retained by the locked pipeline:")
    display(selected_engineered.head(30))
else:
    print("No new engineered features were selected by the locked pipeline.")

if len(removed_features):
    print("Removed features from the locked pipeline:")
    display(removed_features.head(30))
else:
    print("No features were removed because the locked pipeline uses the fixed X5 feature set.")

,selected_pipeline,selector,selected_features,removed_features,selected_new_engineered_features,removed_new_engineered_features,report_interpretation
0,X5_fixed + Random Forest,none,77,0,0,0,No train-fitted feature selector was adopted. ...


No new engineered features were selected by the locked pipeline.
No features were removed because the locked pipeline uses the fixed X5 feature set.


## 15. Old Vs New Change Analysis

This compares the Week 5 benchmark with the Week 6 locked winner. Improvement must be visible across multiple metrics, not only one headline number.

In [22]:
week5_test_metrics = pd.read_csv(WEEK5_METRICS_PATH) if WEEK5_METRICS_PATH.exists() else pd.DataFrame()

required_week5_metric_columns = {"model", "R2", "MAPE", "MdAPE"}
if required_week5_metric_columns.issubset(week5_test_metrics.columns):
    week5_selected = week5_test_metrics[
        week5_test_metrics["model"].astype(str).str.contains("Selected model", na=False)
    ].copy()
else:
    week5_selected = pd.DataFrame()

old_new_rows = []
if len(week5_selected):
    old = week5_selected.iloc[0]
    old_new_rows.append({
        "comparison": "Week 5 benchmark selected model",
        "population": "legacy in-range June population",
        "pipeline": old["model"],
        "test_R2": old["R2"],
        "test_MAPE": old["MAPE"],
        "test_MdAPE": old["MdAPE"],
        "selected_feature_count": np.nan,
        "directly_comparable_to_week5": True,
    })

old_new_rows.append({
    "comparison": "Week 6 locked winner primary",
    "population": "all eligible June SFR transactions",
    "pipeline": selected_pipeline_name,
    "test_R2": selected_metrics["test_R2"],
    "test_MAPE": selected_metrics["test_MAPE"],
    "test_MdAPE": selected_metrics["test_MdAPE"],
    "selected_feature_count": selected_metrics["selected_feature_count"],
    "directly_comparable_to_week5": False,
})

if "secondary_sensitivity_metrics" in globals() and len(secondary_sensitivity_metrics):
    june_secondary = secondary_sensitivity_metrics[
        secondary_sensitivity_metrics["population"].eq("june_in_training_range")
    ]
    if len(june_secondary):
        secondary = june_secondary.iloc[0]
        old_new_rows.append({
            "comparison": "Week 6 locked winner secondary",
            "population": "train-defined in-range June population",
            "pipeline": selected_pipeline_name,
            "test_R2": secondary["metric_R2"],
            "test_MAPE": secondary["metric_MAPE"],
            "test_MdAPE": secondary["metric_MdAPE"],
            "selected_feature_count": selected_metrics["selected_feature_count"],
            "directly_comparable_to_week5": True,
        })

old_vs_new = pd.DataFrame(old_new_rows)
old_vs_new["delta_mdape_vs_week5_selected"] = np.nan
if len(week5_selected):
    week5_mdape = float(old_vs_new.loc[0, "test_MdAPE"])
    comparable_mask = old_vs_new["directly_comparable_to_week5"].eq(True) & (old_vs_new.index != 0)
    old_vs_new.loc[comparable_mask, "delta_mdape_vs_week5_selected"] = (
        old_vs_new.loc[comparable_mask, "test_MdAPE"].astype(float) - week5_mdape
    )

old_vs_new.to_csv(OUTPUT_DIR / "week6_old_vs_new_test_comparison.csv", index=False)
old_vs_new

,comparison,population,pipeline,test_R2,test_MAPE,test_MdAPE,selected_feature_count,directly_comparable_to_week5,delta_mdape_vs_week5_selected
0,Week 5 benchmark selected model,legacy in-range June population,Selected model: Random Forest,0.878003,0.128153,0.083232,NaN,True,NaN
1,Week 6 locked winner primary,all eligible June SFR transactions,X5_fixed + Random Forest,0.596833,0.148390,0.084821,77.0,False,NaN
2,Week 6 locked winner secondary,train-defined in-range June population,X5_fixed + Random Forest,0.879076,0.128015,0.082500,77.0,True,-0.000732


 ### Why Week 5 and Week 6 Results Differ Slightly

- **Same benchmark specification:** <br>both Week 5 and Week 6 use `X5_fixed + Random Forest` with the same RF hyperparameters and `random_state`.

- **Retraining is required:** <br>Week 6 retrains every candidate inside the updated pipeline so preprocessing, fitting, and evaluation follow one consistent workflow.

- **Retraining is not the main explanation:** <br>with identical transformed inputs, row order, feature order, parameters, and random seed, Random Forest should reproduce the same predictions.

- **Main source of difference:** <br>Week 6 rebuilds the transformed feature matrix with the updated preprocessor, especially categorical missing-value and unknown-category handling.

- **Same June comparison population:** <br>both Week 5 and Week 6 evaluate the same 12,566 in-range June transactions using train-derived price and price-per-sqft limits.

- **Performance is effectively unchanged:** <br> Week 6 has slightly lower R2 but slightly better MAPE and MdAPE.

- **Correct conclusion:** <br> Week 6 reproduces the Week 5 RF benchmark under a cleaner pipeline. The small mixed metric differences reflect preprocessing implementation changes, not meaningful model improvement.

## 16. Price Segment And Error Distribution

Segment analysis checks whether the selected pipeline improves real pricing decisions or only improves average metrics.

In [23]:
def make_prediction_frame(frame, prediction, split_name):
    out = frame[["CloseDate", "close_month", TARGET, "City", "PostalCode", "CountyOrParish", "UnifiedSchoolDistrictName"]].copy()
    out["prediction"] = prediction
    out["error"] = out["prediction"] - out[TARGET]
    out["absolute_error"] = out["error"].abs()
    out["absolute_percentage_error"] = out["absolute_error"] / out[TARGET].replace(0, np.nan)
    out["split"] = split_name
    return out


all_predictions = pd.concat([
    make_prediction_frame(selected_fit["train_frame"], train_pred, "train"),
    make_prediction_frame(selected_fit["validation_frame"], validation_pred, "validation"),
    make_prediction_frame(test, test_pred, "test"),
], ignore_index=True)

test_predictions = all_predictions[all_predictions["split"] == "test"].copy()
price_band_edges = selected_fit["train_frame"][TARGET].quantile(np.linspace(0, 1, 6)).to_numpy(copy=True)
price_band_edges[0] = -np.inf
price_band_edges[-1] = np.inf
price_band_edges = np.unique(price_band_edges)
price_segment_labels = ["Q1_lowest", "Q2", "Q3", "Q4", "Q5_highest"][: max(len(price_band_edges) - 1, 0)]
if len(price_band_edges) < 3:
    raise ValueError("Not enough unique training prices to define stable price bands.")

def assign_train_defined_price_segment(values):
    return pd.cut(
        values,
        bins=price_band_edges,
        labels=price_segment_labels,
        include_lowest=True,
    )

test_predictions["price_segment"] = assign_train_defined_price_segment(test_predictions[TARGET])

print("Prepared train, validation, and June prediction frames with train-defined price bands.")


Prepared train, validation, and June prediction frames with train-defined price bands.


In [24]:
segment_errors = (
    test_predictions
    .groupby("price_segment", observed=True)
    .agg(
        rows=(TARGET, "size"),
        median_close_price=(TARGET, "median"),
        mae=("absolute_error", "mean"),
        mape=("absolute_percentage_error", "mean"),
        mdape=("absolute_percentage_error", "median"),
        p90_ape=("absolute_percentage_error", lambda s: s.quantile(0.90)),
    )
    .reset_index()
)

error_distribution = pd.DataFrame([{
    "pipeline": selected_pipeline_name,
    "mean_error": test_predictions["error"].mean(),
    "median_error": test_predictions["error"].median(),
    "p10_error": test_predictions["error"].quantile(0.10),
    "p90_error": test_predictions["error"].quantile(0.90),
    "p50_ape": test_predictions["absolute_percentage_error"].quantile(0.50),
    "p75_ape": test_predictions["absolute_percentage_error"].quantile(0.75),
    "p90_ape": test_predictions["absolute_percentage_error"].quantile(0.90),
    "p95_ape": test_predictions["absolute_percentage_error"].quantile(0.95),
}])

all_predictions.to_csv(OUTPUT_DIR / "week6_selected_pipeline_predictions.csv", index=False)
test_predictions.to_csv(OUTPUT_DIR / "week6_selected_pipeline_test_predictions.csv", index=False)
segment_errors.to_csv(OUTPUT_DIR / "week6_selected_pipeline_segment_errors.csv", index=False)
error_distribution.to_csv(OUTPUT_DIR / "week6_selected_pipeline_error_distribution.csv", index=False)

pd.DataFrame({
    "price_segment": price_segment_labels,
    "lower_bound": price_band_edges[:-1],
    "upper_bound": price_band_edges[1:],
    "source": "training ClosePrice quantiles",
}).to_csv(OUTPUT_DIR / "week6_train_defined_price_bands.csv", index=False)

print("Saved Week 6 segment errors and error distribution.")


Saved Week 6 segment errors and error distribution.


In [25]:
if WEEK5_TEST_PREDICTIONS_PATH.exists():
    week5_test_predictions = pd.read_csv(WEEK5_TEST_PREDICTIONS_PATH)
    required_week5_prediction_columns = {TARGET, "absolute_percentage_error", "error"}
    if required_week5_prediction_columns.issubset(week5_test_predictions.columns):
        week5_test_predictions["price_segment"] = assign_train_defined_price_segment(
            pd.to_numeric(week5_test_predictions[TARGET], errors="coerce")
        )
        week5_segment_errors = (
            week5_test_predictions
            .groupby("price_segment", observed=True)
            .agg(
                rows=(TARGET, "size"),
                median_close_price=(TARGET, "median"),
                mae=("error", lambda s: s.abs().mean()),
                mape=("absolute_percentage_error", "mean"),
                mdape=("absolute_percentage_error", "median"),
                p90_ape=("absolute_percentage_error", lambda s: s.quantile(0.90)),
            )
            .reset_index()
        )
    else:
        week5_segment_errors = pd.DataFrame()
elif WEEK5_SEGMENT_PATH.exists():
    week5_segment_errors = pd.read_csv(WEEK5_SEGMENT_PATH)
else:
    week5_segment_errors = pd.DataFrame()

if len(week5_segment_errors):
    segment_comparison = week5_segment_errors.merge(
        segment_errors,
        on="price_segment",
        suffixes=("_week5", "_week6"),
    )
    for metric in ["mae", "mape", "mdape", "p90_ape"]:
        segment_comparison[f"{metric}_delta_week6_minus_week5"] = (
            segment_comparison[f"{metric}_week6"] - segment_comparison[f"{metric}_week5"]
        )
else:
    segment_comparison = pd.DataFrame()

segment_comparison.to_csv(OUTPUT_DIR / "week6_segment_old_vs_new_comparison.csv", index=False)

display(segment_errors)
display(segment_comparison)
display(error_distribution)

,price_segment,rows,median_close_price,mae,mape,mdape,p90_ape
0,Q1_lowest,2614,449000.0,69105.943185,0.218081,0.082104,0.478263
1,Q2,2361,688888.0,71575.566344,0.105653,0.061349,0.238924
2,Q3,2436,900000.0,103521.272119,0.113052,0.069747,0.234872
3,Q4,2583,1285000.0,167021.192883,0.128565,0.089435,0.260278
4,Q5_highest,2857,2265000.0,688215.698195,0.167998,0.126124,0.360636


,price_segment,rows_week5,median_close_price_week5,mae_week5,mape_week5,mdape_week5,p90_ape_week5,rows_week6,median_close_price_week6,mae_week6,mape_week6,mdape_week6,p90_ape_week6,mae_delta_week6_minus_week5,mape_delta_week6_minus_week5,mdape_delta_week6_minus_week5,p90_ape_delta_week6_minus_week5
0,Q1_lowest,2483,451500.0,61178.987752,0.147742,0.077561,0.379644,2614,449000.0,69105.943185,0.218081,0.082104,0.478263,7926.955433,0.070339,0.004543,0.098619
1,Q2,2354,688944.0,68543.596321,0.100906,0.060867,0.234984,2361,688888.0,71575.566344,0.105653,0.061349,0.238924,3031.970022,0.004747,0.000482,0.003940
2,Q3,2434,900000.0,100546.968975,0.110037,0.069860,0.241280,2436,900000.0,103521.272119,0.113052,0.069747,0.234872,2974.303144,0.003014,-0.000113,-0.006408
3,Q4,2581,1285000.0,165694.966388,0.127623,0.090730,0.263684,2583,1285000.0,167021.192883,0.128565,0.089435,0.260278,1326.226495,0.000942,-0.001295,-0.003405
4,Q5_highest,2714,2220000.0,422830.031127,0.150615,0.121827,0.313554,2857,2265000.0,688215.698195,0.167998,0.126124,0.360636,265385.667069,0.017383,0.004296,0.047082


,pipeline,mean_error,median_error,p10_error,p90_error,p50_ape,p75_ape,p90_ape,p95_ape
0,X5_fixed + Random Forest,-72343.751858,4046.262412,-262336.190792,226254.256454,0.084821,0.173538,0.312985,0.455324


### Price Segment Interpretation

- Best performance is in the middle price ranges.
- The model is most useful for typical mid-market homes.
- Manual review is still required for low-price outliers and luxury properties.

## 17. Strengths And Weaknesses By Pipeline Type

### Fixed X5 Pipelines

**Strengths:**
- Fair benchmark because it matches Week 5 feature logic.
- Easier to explain because the feature list is fixed before modeling.

**Weaknesses:**
- Does not reduce redundant transformed variables.
- May miss useful engineered signals like district frequency or tax intensity.

### Linear Regularization Pipelines

**Strengths:**
- Better for many correlated variables than plain Linear Regression.
- Lasso-selected features give a compact, interpretable pricing feature list.

**Weaknesses:**
- Still limited by mostly linear relationships.
- Can underuse nonlinear location and school-district interactions.

### Tree-Based Selected Pipelines

**Strengths:**
- Better fit for nonlinear real estate patterns across property, geography, and school districts.
- More useful for valuation triage if segment errors improve.

**Weaknesses:**
- Less transparent for stakeholder explanation.
- Needs Q1/Q5 and tail-error checks before pricing use.

## 18. Save Reproducibility Artifacts

In [26]:
model_artifact = {
    "pipeline": selected_fit["raw_prediction_pipeline"],
    "model": selected_fit["model"],
    "preprocessor": selected_fit["preprocessor"],
    "preprocess_params": selected_fit["preprocess_params"],
    "selected_features": selected_fit["selected_features"],
    "removed_features": selected_fit["removed_features"],
    "candidate": selected_fit["candidate"],
    "target": TARGET,
    "outlier_params": outlier_params,
    "raw_prediction_contract": "Pass raw CRMLS-like property rows to artifact['pipeline'].predict(raw_rows).",
}
joblib.dump(model_artifact, OUTPUT_DIR / "week6_locked_pipeline.joblib")

loaded_artifact = joblib.load(OUTPUT_DIR / "week6_locked_pipeline.joblib")
raw_artifact_check_rows = pd.read_csv(DATA_PATH, low_memory=False)
raw_artifact_check_rows["CloseDate"] = pd.to_datetime(raw_artifact_check_rows["CloseDate"], errors="coerce")
raw_artifact_check_rows["close_month"] = pd.PeriodIndex(raw_artifact_check_rows["close_month"].astype(str), freq="M").astype(str)
sample_raw_rows = (
    raw_artifact_check_rows[
        raw_artifact_check_rows["PropertyType"].astype(str).str.strip().eq("Residential")
        & raw_artifact_check_rows["PropertySubType"].astype(str).str.strip().eq("SingleFamilyResidence")
        & raw_artifact_check_rows["close_month"].eq(TEST_MONTH)
    ]
    .head(5)
    .drop(columns=[TARGET], errors="ignore")
)
sample_artifact_predictions = loaded_artifact["pipeline"].predict(sample_raw_rows)
if len(sample_artifact_predictions) != len(sample_raw_rows):
    raise ValueError("Saved raw pipeline prediction check failed.")

metadata = {
    "core_rule": "Train creates candidate pipelines; validation chooses among completed pipelines; test evaluates the locked winner.",
    "selected_pipeline": selected_pipeline_name,
    "selected_metrics": selected_metrics,
    "split": {
        "train_start_month": TRAIN_START_MONTH,
        "train_end_month": TRAIN_END_MONTH,
        "validation_month": VALIDATION_MONTH,
        "test_month": TEST_MONTH,
    },
    "candidate_pipeline_count": len(CANDIDATE_PIPELINES),
    "selected_feature_count": len(selected_fit["selected_features"]),
    "removed_feature_count": len(selected_fit["removed_features"]),
}

with open(OUTPUT_DIR / "week6_locked_pipeline_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved Week 6 outputs to:", OUTPUT_DIR)
print("Saved raw pipeline artifact prediction check rows:", len(sample_artifact_predictions))

Saved Week 6 outputs to: /Users/amyliu/Desktop/summer intern/outputs/week6_feature_engineering
Saved raw pipeline artifact prediction check rows: 5


## 19. Final Report Interpretation

### Executive Conclusion

- Selected pipeline: `X5_fixed + Random Forest`.
- Week 6 keeps the Week 5 model structure: fixed X5 features plus Random Forest.
- Primary June test uses all eligible June SFR sales, not a ClosePrice-filtered subset.
- Primary June performance: R2 `0.5968`, MAPE `14.84%`, MdAPE `8.48%`.
- Secondary in-range performance: R2 `0.8791`, MAPE `12.80%`, MdAPE `8.25%`.
- Comparable in-range MdAPE is `0.07 bp` better than Week 5.

### Controlled Feature-Set Findings

- School layer improves X5 RF: MdAPE `-0.28 bp`.
- School layer improves X6 RF: MdAPE `-0.22 bp`.
- X6 full does not beat X5 full: MdAPE `+0.03 bp`.
- RF selection does not improve X6 full: MdAPE `+0.06 bp`.
- Decision: keep X5 RF. X6 full and X6 selected do not justify replacement.

### Why X6 Did Not Win

- Most X6 features repackage existing X5 signals.
- Random Forest already captures many nonlinear X5 interactions.
- Ratio features can add noise when MLS fields are sparse or inconsistently recorded.
- Feature selection may remove weak standalone variables that still help through interactions.
- Conclusion is limited: this X6 design did not beat X5 RF in validation.

### Evaluation Caveat

- June is a repeated comparison month, not a fully untouched final holdout.
- Primary June metrics are stricter because they include all eligible sales.
- R2 drops mainly because extreme June transactions remain in the primary test set.
- Final model confidence still requires a future unseen month or rolling-origin backtesting.

### Segment Risk

- Q1 low-price homes: MdAPE `8.21%`, P90 APE `47.83%`.
- Q5 high-price homes: MdAPE `12.61%`, P90 APE `36.06%`.
- Manual review is still required for low-price outliers and luxury homes.

### Business Decision

- Listing review: use predictions as directional pricing support.
- Valuation triage: prioritize high-risk records for manual review.
- Week 7 focus: rolling-origin validation, prediction-time features, and segment reliability.
